# 自定义中间件-Wrap-style hooks
## 1.wrap_model_call的使用
### 1.1基于装饰器的实现

In [1]:
import os

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, PIIMiddleware, TodoListMiddleware, wrap_model_call, \
    ModelRequest, ModelResponse
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from openai.types.beta.realtime import response_text_done_event
from scripts.regsetup import description
from rich import print as rprint
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    # profile={
    #     "max_input_tokens": 1_000_000
    # },
    # 关键修改：关闭思考模式
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    },
)


In [5]:
from typing import Callable
from langchain.agents.middleware import wrap_model_call,ModelRequest,ModelResponse

@wrap_model_call
def wrap_model_call_middleware(
        request:ModelRequest,
        hanler:Callable[[ModelRequest], ModelResponse],
)->ModelResponse|None:
    request.messages[-1].content+="---->wrap_model_call_before<----"
    #模型的调用
    response=hanler(request)

    response.result[0].content+="---->wrap_model_call_after<----"
    return response


In [7]:
agent=create_agent(
    model=model,
    middleware=[wrap_model_call_middleware]
)
response=agent.invoke({
    "messages":[HumanMessage("你好")]
})
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好---->wrap_model_call_before<----
================================== Ai Message ==================================

你好！👋 很高兴见到你！

我是DeepSeek，一个由深度求索公司创造的AI助手。我可以帮你回答问题、提供建议、处理文档、进行头脑风暴等等。无论是学习、工作还是生活中的问题，我都乐意为你提供帮助！

有什么我可以为你做的吗？尽管说吧，我会尽力为你解答！😊---->wrap_model_call_after<----


### 1.2基于类的实现

In [8]:
from langchain.agents.middleware import AgentMiddleware


class WrapModelCallMiddleware(AgentMiddleware):
    def wrap_model_call(self, request: ModelRequest,
                        handler: Callable[[ModelRequest], ModelResponse],
                        ) -> ModelResponse | None:
        request.messages[-1].content += "---> wrap_model_call_before <---"

        # 模型的调用
        response = handler(request)

        response.result[0].content += "---> wrap_model_call_after <---"

        return response


agent = create_agent(
    model=model,
    middleware=[
        WrapModelCallMiddleware(),
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("你好")]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好---> wrap_model_call_before <---
================================== Ai Message ==================================

你好！很高兴见到你。今天有什么我可以帮助你的吗？无论是问题、聊天，还是需要协助处理某些事项，我都在这里。😊---> wrap_model_call_after <---
